# exp01 — Baseline

| 항목 | 값 |
|---|---|
| config | `experiments/configs/exp01_baseline.yaml` |
| 결과 저장 | `experiments/results/exp01_baseline/` |
| 핵심 세팅 | hidden=128, layers=2, heads=4, lr_w=0.005, lr_α=0.005, temperature=1.0, complement=False |
| 목적 | `docs/training_decisions.md` 확정 세팅의 재현·비교 기준점 확보 |
| 기존 결과 | test PR-AUC=0.4844, AUC-ROC=0.7789 (α 균등 → 미분화 상태) |

In [ ]:
import os, sys
# 프로젝트 루트를 Python 경로에 추가
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글
matplotlib.rcParams['axes.unicode_minus'] = False

from experiments.exp_utils import (
    run_experiment, load_experiment, compare_experiments,
    plot_alpha_heatmap, plot_training_curve, print_metrics_table, print_recommendations
)

EXP_NAME   = 'exp01_baseline'
CFG_PATH   = 'experiments/configs/exp01_baseline.yaml'
print('ROOT:', ROOT)

## 1. 학습 실행

> 이미 체크포인트가 있으면 export만 수행. 재학습 강제: `force=True`

In [ ]:
results = run_experiment(CFG_PATH, EXP_NAME)
# 재학습 강제: results = run_experiment(CFG_PATH, EXP_NAME, force=True)

## 2. 성능 지표

In [ ]:
print_metrics_table(results)
# 양성 23.6% → 랜덤 PR-AUC = 0.236. test PR-AUC가 그 2배(~0.47↑)면 유의미.

## 3. 학습 곡선 (val PR-AUC)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(results.get('history', []), ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. DiffMG α_r — 관계 중요도 히트맵

> 모든 값이 ≈1/R(균등)이면 게이팅 미분화 상태. exp02에서 lr_α 인상으로 개선 예정.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
plot_alpha_heatmap(EXP_NAME, ax=ax)
plt.savefig(f'experiments/results/{EXP_NAME}/alpha_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 순회 추천 샘플 (시드 → 조합 키워드 top10)

In [ ]:
print_recommendations(results)

## 6. 실험 간 비교 (현재까지 완료된 실험)

In [ ]:
df = compare_experiments(['exp01_baseline', 'exp02_alpha_tuning', 'exp03_complement_edges'])
display(df[['exp', 'val_pr_auc', 'val_auc_roc', 'test_pr_auc', 'test_auc_roc', 'test_f1']])